In [18]:
import os
import numpy as np
# uncomment to disable NVIDIA GPUs
#os.environ['CUDA_VISIBLE_DEVICES'] = ''
# or pick the device (cpu, gpu, and tpu)
#os.environ['JAX_PLATFORMS'] = 'cpu'

# change JAX GPU memory preallocation fraction
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.95'

# you do not want this
#os.environ['XLA_FLAGS'] = '--xla_gpu_deterministic_ops=true'

import jax
import jax.numpy as jnp
from jax import jit, lax
#jax.print_environment_info()

#!nvidia-smi --query-gpu=gpu_name --format=csv,noheader

import matplotlib.pyplot as plt
# matplotlib de casa
#import matplotlib_inline
#matplotlib_inline.backend_inline.set_matplotlib_formats('jpeg')

import Pk_library as PKL

from pmwd import (
    Configuration,
    Cosmology, SimpleLCDM,
    boltzmann, linear_power, growth,
    white_noise, linear_modes,
    lpt,
    nbody,
    scatter,
)
from pmwd.nbody import nbody_step, nbody_init
from pmwd.pm_util import fftinv
from pmwd.spec_util import powspec
from pmwd.vis_util import simshow

In [19]:
if jax.default_backend() == 'gpu':
    ptcl_spacing = 1.  # Lagrangian space Cartesian particle grid spacing, in Mpc/h by default
    ptcl_grid_shape = (128,) * 3
else:
    ptcl_spacing = 4.
    #ptcl_grid_shape = (64,) * 3
    ptcl_grid_shape = (128,) * 3

conf = Configuration(ptcl_spacing, ptcl_grid_shape, mesh_shape=2)  # 2x mesh shape

print(conf)  # with other default parameters
print(f'\n Simulating {conf.ptcl_num} particles with a {conf.mesh_shape} mesh for {conf.a_nbody_num} time steps.')

Configuration(ptcl_spacing=4.0,
              ptcl_grid_shape=(128, 128, 128),
              mesh_shape=(256, 256, 256),
              cosmo_dtype=dtype('float64'),
              pmid_dtype=dtype('int16'),
              float_dtype=dtype('float32'),
              k_pivot_Mpc=0.05,
              T_cmb=2.7255,
              M=1.98847e+40,
              L=3.0856775815e+22,
              T=3.0856775815e+17,
              transfer_fit=True,
              transfer_fit_nowiggle=False,
              transfer_lgk_min=-4,
              transfer_lgk_max=3,
              transfer_lgk_maxstep=0.0078125,
              growth_rtol=1.4901161193847656e-08,
              growth_atol=1.4901161193847656e-08,
              growth_inistep=(1, None),
              lpt_order=2,
              a_start=0.015625,
              a_stop=1,
              a_lpt_maxstep=0.0078125,
              a_nbody_maxstep=0.015625,
              symp_splits=((0, 0.5), (1, 0.5)),
              chunk_size=16777216)

 Simulating 2097

In [20]:
def phase_space(modes, cosmo, conf, output_dir='phase_space_data'):
    """
    salva o espaço de fase e o espectro de potência.
    Pk foi com o pylians q nem a documentação.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    omega_m = float(cosmo.Omega_m)
    h = float(cosmo.h)
    cosmo_data = f'Omega_m_{omega_m:.2f}_h_{h:.2f}'

    cosmo = jax.block_until_ready(boltzmann(cosmo, conf))
    modes_lin = linear_modes(modes, cosmo, conf)
    ptcl, obsvbl = jax.block_until_ready(lpt(modes_lin, cosmo, conf))
    ptcl, obsvbl = jax.block_until_ready(nbody_init(conf.a_nbody[0], ptcl, obsvbl, cosmo, conf))
    for a_prev, a_next in zip(conf.a_nbody[:-1], conf.a_nbody[1:]):
        ptcl, obsvbl = jax.block_until_ready(
            nbody_step(a_prev, a_next, ptcl, obsvbl, cosmo, conf)
        )

    dens = scatter(ptcl, conf)
    dens_array = np.array(dens)
    delta = dens_array / np.mean(dens_array) - 1.0
    
    box_size = float(conf.box_size[0])
    mesh_shape = conf.mesh_shape
    
    Pk = PKL.Pk(delta, box_size, axis=0, MAS='CIC', threads=1, verbose=False)
    k_pk = np.column_stack([Pk.k3D, Pk.Pk[:,0]])
    
    pk_file = f'pk_pylians_{cosmo_data}.txt'
    np.savetxt(os.path.join(output_dir, pk_file), 
               k_pk, fmt='%.5e', 
               header='k [h/Mpc]    P(k) [(Mpc/h)^3]')
    mesh = jnp.zeros(tuple(2*s for s in conf.mesh_shape), dtype=conf.float_dtype)
    phase = scatter(ptcl, conf, mesh=mesh, val=1, cell_size=conf.cell_size/2)
    phase_2d = phase.sum(axis=2)
    
    phase_file = f'phase_space_{cosmo_data}.txt'
    np.savetxt(os.path.join(output_dir, phase_file), np.array(phase_2d), fmt='%.5e')
    
    info_file = f'simulation_info_{cosmo_data}.txt'
    with open(os.path.join(output_dir, info_file), 'w') as f:
        f.write("Simulation With these parameters\n")
        f.write("------------------------\n\n")
        f.write("Configurations:\n")
        f.write(f"box_size = {box_size}\n")
        f.write(f"mesh_shape = {mesh_shape}\n")
        f.write("Cosmological Parameters:\n")
        f.write(f"A_s_1e9 = {float(cosmo.A_s_1e9)}\n")
        f.write(f"n_s = {float(cosmo.n_s)}\n")
        f.write(f"Omega_m = {omega_m}\n")
        f.write(f"Omega_b = {float(cosmo.Omega_b)}\n")
        f.write(f"h = {h}\n")
        f.write(f"xi = {float(cosmo.xi)}\n")
    print(f'salvo na pasta {output_dir}/ o Pk de {cosmo_data}:')
    print(f'  - Espaço de fase: {phase_file}')
    print(f'  - Espectro de potência: {pk_file}')
    print(f'  - Dados: {info_file}')

In [21]:

cosmo = Cosmology(conf, A_s_1e9=2.1, n_s=0.96, Omega_m=0.30, Omega_b=0.05, h=0.67, xi_=0.0, w_0_=-1.0, w_a_= 0.0)
modes = white_noise(1, conf)

phase_space(modes, cosmo, conf, output_dir='phase_space_data')


salvo na pasta phase_space_data/ o Pk de Omega_m_0.30_h_0.67:
  - Espaço de fase: phase_space_Omega_m_0.30_h_0.67.txt
  - Espectro de potência: pk_pylians_Omega_m_0.30_h_0.67.txt
  - Dados: simulation_info_Omega_m_0.30_h_0.67.txt
